# Phase 4 — baseline table

The ablation showed tafsir hurts, so every system here indexes **verses only**.

Adds the two baselines the project never had:

- **base GATE-AraBert-v1**, un-fine-tuned — did fine-tuning help at all?
- **BM25**, lexical — the standard IR baseline
- **hybrid RRF** — fusion of dense + lexical

**T4 GPU → Run all.** ~8 minutes.

### 1. Setup

In [ ]:
# ---- Cell 1: setup (safe to re-run) ----
import os, sys, shutil, subprocess, json, time, math, pickle, re
import numpy as np, statistics as st
import torch
from google.colab import drive
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Enable T4: Runtime -> Change runtime type -> T4 GPU")
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

ROOT="/content/drive/MyDrive"
P1=f"{ROOT}/Phase1_Project/MemberB_B4_B6_output"
P1FIX=f"{ROOT}/Phase1_Project/data_fix_output"
GV2=f"{ROOT}/Phase3_Project/guardrail_output_v2"
OUT=f"{ROOT}/Phase4_Project"; os.makedirs(f"{OUT}/data", exist_ok=True)
ART="/content/artifacts"; PROJECT="/content/QuranicRAG"
os.makedirs(ART, exist_ok=True)
os.makedirs(f"{PROJECT}/src", exist_ok=True)
os.makedirs(f"{PROJECT}/quranNLP/shared/data", exist_ok=True)
os.chdir(PROJECT)

shutil.rmtree("/content/_repo", ignore_errors=True)
subprocess.run(["git","clone","--depth","1",
  "https://github.com/Laiba-Noor/quranic-rag-hallucination-free.git","/content/_repo"],check=True)
for s in ["/content/_repo/src", f"{GV2}/src"]:
    for f in os.listdir(s):
        if f.endswith(".py"): shutil.copy(f"{s}/{f}", f"{PROJECT}/src/{f}")

CSV=f"{PROJECT}/quranNLP/shared/data/final_cross_reference_index.csv"
if not os.path.exists(CSV):
    shutil.copy(f"{P1FIX}/shared_data/final_cross_reference_index.csv", CSV)

try:
    import hnswlib, sentence_transformers, scipy, rank_bm25   # noqa
    print("deps present")
except ImportError:
    !pip install -q sentence-transformers hnswlib scipy rank_bm25
print("SETUP OK")

### 2. Models, corpus, ground truth

In [ ]:
# ---- Cell 2: models (skips what is already local) ----
for local, remote in {"m_v1": f"{P1}/b5_real_finetuned",
                      "m_v2": f"{GV2}/b5_real_finetuned_v2"}.items():
    dst=f"{ART}/{local}"
    if os.path.isdir(dst) and os.listdir(dst): print("  have", local); continue
    print("  copying", local, "..."); shutil.copytree(remote, dst, dirs_exist_ok=True)

sys.path.insert(0, f"{PROJECT}/src")
from sentence_transformers import SentenceTransformer
import hnswlib

# corpus: verses only - the ablation showed tafsir hurts
import csv; csv.field_size_limit(sys.maxsize)
rows=list(csv.DictReader(open(CSV, encoding="utf-8")))
VERSES=[(r["verse_key"], (r.get("clean_verse") or "").strip())
        for r in rows if (r.get("clean_verse") or "").strip()]
print(f"{len(VERSES)} verses")

STASH=f"{OUT}/data/ayatec_records.json"
if not os.path.exists(STASH):
    for c in ["/content/_repo/Data/ayatec_records.json",
              f"{ROOT}/Phase2_Project/Roma_output/data/ayatec_records.json"]:
        if os.path.exists(c): shutil.copy(c, STASH); break
if not os.path.exists(STASH):
    from google.colab import files
    print("Upload ayatec_records.json"); up=files.upload()
    shutil.copy(list(up.keys())[0], STASH)
aya=json.load(open(STASH, encoding="utf-8"))
GOLD=[(r["question"], set(r["verse_keys"])) for r in aya
      if r.get("question") and r.get("verse_keys")]
ceil10=sum(min(10,len(g))/len(g) for _,g in GOLD)/len(GOLD)
print(f"{len(GOLD)} questions | ceiling Recall@10 = {ceil10:.4f}")

### 3. Metrics

In [ ]:
# ---- Cell 3: metrics (dense + lexical share one interface) ----
def metrics_from_ranked(ranked, gold, ks=(1,5,10,20)):
    out={}
    out["MRR"]=next((1.0/i for i,vk in enumerate(ranked,1) if vk in gold), 0.0)
    dcg=sum(1/math.log2(i+1) for i,vk in enumerate(ranked[:10],1) if vk in gold)
    idcg=sum(1/math.log2(i+1) for i in range(1,min(len(gold),10)+1))
    out["NDCG@10"]=dcg/idcg if idcg else 0.0
    for k in ks:
        h=sum(1 for vk in ranked[:k] if vk in gold)
        out[f"Recall@{k}"]=h/len(gold); out[f"HitRate@{k}"]=1.0 if h else 0.0
    return out

def evaluate_ranker(rank_fn, gold_pairs, want=30):
    from collections import defaultdict
    acc=defaultdict(list); per_q={"r10":[], "hit20":[], "rr":[]}
    for q,gold in gold_pairs:
        ranked=rank_fn(q, want)
        m=metrics_from_ranked(ranked, gold)
        for k,v in m.items(): acc[k].append(v)
        per_q["rr"].append(m["MRR"])
        per_q["r10"].append(m["Recall@10"])
        per_q["hit20"].append(m["HitRate@20"])
    return {k: sum(v)/len(v) for k,v in acc.items()}, per_q

def show(title, rows):
    print(f"\n{title}")
    print(f"{'system':<30}{'R@10':>8}{'R@20':>8}{'Hit@10':>9}{'Hit@20':>9}{'MRR':>8}{'NDCG':>8}")
    print("-"*80)
    for n,r in rows:
        print(f"{n:<30}{r['Recall@10']:>8.4f}{r['Recall@20']:>8.4f}"
              f"{r['HitRate@10']:>9.4f}{r['HitRate@20']:>9.4f}{r['MRR']:>8.4f}{r['NDCG@10']:>8.4f}")

def compare(label, a, b, keys=("r10","hit20")):
    from scipy.stats import wilcoxon
    for kk in keys:
        x,y=a[kk],b[kk]; d=[q-p for p,q in zip(x,y)]
        if not any(d): print(f"{label} {kk:<6} identical"); continue
        _,p=wilcoxon(x,y,zero_method="wilcox"); sd=st.pstdev(d) or 1e-9
        print(f"{label} {kk:<6} delta={st.mean(d):+.4f} p={p:.4g} "
              f"d={st.mean(d)/sd:+.3f} {'SIGNIFICANT' if p<0.05 else 'ns'}")
print("metrics ready")

### 4. Dense retrievers

In [ ]:
# ---- Cell 4: three dense models, verses-only index each ----
def build_dense(model_path, tag):
    m=SentenceTransformer(model_path); m.max_seq_length=64
    emb=m.encode([t for _,t in VERSES], convert_to_numpy=True, normalize_embeddings=True,
                 show_progress_bar=True, batch_size=256).astype(np.float32)
    ix=hnswlib.Index(space="cosine", dim=emb.shape[1])
    ix.init_index(max_elements=len(VERSES), ef_construction=200, M=16)
    ix.add_items(emb, ids=np.arange(len(VERSES)))
    keys=[k for k,_ in VERSES]
    def rank(q, want=30):
        k=min(max(want*2, 64), len(VERSES)-1)
        while k>=1:
            try:
                ix.set_ef(min(max(k*2,64), len(VERSES)))
                lab,_=ix.knn_query(m.encode([q], convert_to_numpy=True,
                                            normalize_embeddings=True), k=k)
                break
            except RuntimeError:
                k//=2
        else:
            return []
        seen,out=set(),[]
        for i in lab[0]:
            vk=keys[i]
            if vk not in seen: seen.add(vk); out.append(vk)
            if len(out)>=want: break
        return out
    print(f"  {tag}: index built ({len(VERSES)} verses)")
    return rank

t=time.time()
RANKERS={}
RANKERS["base GATE-AraBert-v1 (no fine-tune)"] = build_dense(
    "Omartificial-Intelligence-Space/GATE-AraBert-v1", "base")
RANKERS["v1 fine-tuned"] = build_dense(f"{ART}/m_v1", "v1")
RANKERS["v2 fine-tuned (+AyaTEC)"] = build_dense(f"{ART}/m_v2", "v2")
print(f"built in {(time.time()-t)/60:.1f} min")

### 5. BM25

In [ ]:
# ---- Cell 5: BM25 lexical baseline ----
from rank_bm25 import BM25Okapi

AR_DIAC = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670\u06D6-\u06ED\u0640]")
def norm(t):
    t = AR_DIAC.sub("", t)
    t = re.sub(r"[إأآٱ]", "ا", t)
    t = re.sub(r"ى", "ي", t); t = re.sub(r"ة", "ه", t)
    t = re.sub(r"[^\u0600-\u06FF\s]", " ", t)
    return t.split()

corpus=[norm(t) for _,t in VERSES]
keys=[k for k,_ in VERSES]
bm25=BM25Okapi(corpus)
print(f"BM25 over {len(corpus)} verses | avg {sum(len(c) for c in corpus)/len(corpus):.1f} tokens/verse")

def bm25_rank(q, want=30):
    scores=bm25.get_scores(norm(q))
    order=np.argsort(-scores)[:want*3]
    seen,out=set(),[]
    for i in order:
        vk=keys[i]
        if vk not in seen: seen.add(vk); out.append(vk)
        if len(out)>=want: break
    return out
RANKERS["BM25 lexical"]=bm25_rank
print("BM25 ready")

### 6. Hybrid

In [ ]:
# ---- Cell 6: hybrid — reciprocal rank fusion of BM25 + best dense ----
def rrf(rankers, k=60):
    def rank(q, want=30):
        scores={}
        for r in rankers:
            for i,vk in enumerate(r(q, want*2), 1):
                scores[vk]=scores.get(vk,0.0)+1.0/(k+i)
        return [vk for vk,_ in sorted(scores.items(), key=lambda x:-x[1])][:want]
    return rank

RANKERS["hybrid RRF (v2 + BM25)"]=rrf([RANKERS["v2 fine-tuned (+AyaTEC)"],
                                        RANKERS["BM25 lexical"]])
print("hybrid ready")

### 7. Results

In [ ]:
# ---- Cell 7: evaluate everything ----
res, per = {}, {}
for name, fn in RANKERS.items():
    t=time.time()
    res[name], per[name] = evaluate_ranker(fn, GOLD)
    print(f"  {name:<40} {time.time()-t:>5.1f}s")

order=["base GATE-AraBert-v1 (no fine-tune)","v1 fine-tuned","v2 fine-tuned (+AyaTEC)",
       "BM25 lexical","hybrid RRF (v2 + BM25)"]
show("PHASE 4 BASELINE TABLE", [(n,res[n]) for n in order])
print(f"\n(max achievable Recall@10 = {ceil10:.4f})\n")

base="base GATE-AraBert-v1 (no fine-tune)"
compare("v1  vs base   ", per[base], per["v1 fine-tuned"])
compare("v2  vs base   ", per[base], per["v2 fine-tuned (+AyaTEC)"])
compare("BM25 vs v2    ", per["v2 fine-tuned (+AyaTEC)"], per["BM25 lexical"])
compare("hybrid vs v2  ", per["v2 fine-tuned (+AyaTEC)"], per["hybrid RRF (v2 + BM25)"])

json.dump({"n_questions":len(GOLD),"ceiling_recall10":ceil10,"results":res},
          open(f"{OUT}/phase4_baselines.json","w"), indent=2, ensure_ascii=False)
print("\nsaved:", f"{OUT}/phase4_baselines.json")